In [4]:
# import pandas as pd

# df = pd.read_csv("comparison_results_v2.csv")

# # Direction of actual return
# actual_dir = df["Actual"].diff() > 0

# # Direction of predicted movements relative to yesterday's actual
# # If predicted > previous day's actual price, it predicted an UP day
# top_down_dir = df["TopDown_Pred"] > df["Actual"].shift(1)
# bottom_up_dir = df["BottomUp_Pred"] > df["Actual"].shift(1)

# mda_A = (actual_dir == top_down_dir).mean() * 100
# mda_B = (actual_dir == bottom_up_dir).mean() * 100

# print(f"Model A (Top-Down Baseline) Directional Accuracy: {mda_A:.2f}%")
# print(f"Model B (Bottom-Up Top 5)   Directional Accuracy: {mda_B:.2f}%")

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# 1. File paths
BASE_FILE   = "top_down_results.csv"
NORM_FILE   = "comparison_results(normal).csv"
UNNORM_FILE = "comparison_results_v2(unnormal).csv"

# 2. Load CSVs with Date index
df_base   = pd.read_csv(BASE_FILE, parse_dates=["Date"], index_col="Date")
df_norm   = pd.read_csv(NORM_FILE, parse_dates=["Date"], index_col="Date")
df_unnorm = pd.read_csv(UNNORM_FILE, parse_dates=["Date"], index_col="Date")

# 3. Inner join on Date, renaming 'BottomUp_Pred' from each file to distinguish them
df = df_base[["Actual", "Pred_TopDown"]].join([
    df_norm[["BottomUp_Pred"]].rename(columns={"BottomUp_Pred": "BottomUp_Norm"}),
    df_unnorm[["BottomUp_Pred"]].rename(columns={"BottomUp_Pred": "BottomUp_Unnorm"})
], how="inner").dropna()

# 4. Shift Actual by 1 day to get yesterday's actual close (P_t)
df["Prev_Actual"] = df["Actual"].shift(1)
df_eval = df.dropna().copy()

actual_next = df_eval["Actual"].values
prev_actual = df_eval["Prev_Actual"].values
actual_dir  = actual_next > prev_actual

models = {
    "1. Top-Down (Baseline)":      df_eval["Pred_TopDown"].values,
    "2. Bottom-Up (Normalized)":   df_eval["BottomUp_Norm"].values,
    "3. Bottom-Up (Unnormalized)": df_eval["BottomUp_Unnorm"].values,
}

# 5. Print 3-Way Scorecard (RMSE, MAPE, MDA)
print("=" * 72)
print(f"{'Model Approach':<32} | {'RMSE':>9} | {'MAPE':>8} | {'MDA':>8}")
print("=" * 72)

for name, preds in models.items():
    rmse = np.sqrt(mean_squared_error(actual_next, preds))
    mape = mean_absolute_percentage_error(actual_next, preds) * 100
    pred_dir = preds > prev_actual
    mda = (actual_dir == pred_dir).mean() * 100
    print(f"{name:<32} | {rmse:9.2f} | {mape:7.2f}% | {mda:7.2f}%")

print("=" * 72)
print(f"Aligned Out-of-Sample Trading Days: {len(df_eval)}\n")

# 6. Diebold-Mariano (DM) Test vs. Top-Down Baseline
def diebold_mariano(actual, pred_model, pred_baseline):
    d = (actual - pred_model) ** 2 - (actual - pred_baseline) ** 2
    dm_stat = np.mean(d) / np.sqrt(np.var(d, ddof=1) / len(d))
    p_val = 2 * (1 - stats.t.cdf(np.abs(dm_stat), df=len(d) - 1))
    return dm_stat, p_val

dm_norm, p_norm     = diebold_mariano(actual_next, models["2. Bottom-Up (Normalized)"], models["1. Top-Down (Baseline)"])
dm_unnorm, p_unnorm = diebold_mariano(actual_next, models["3. Bottom-Up (Unnormalized)"], models["1. Top-Down (Baseline)"])

print("--- DIEBOLD-MARIANO SIGNIFICANCE TESTS (vs. Top-Down Baseline) ---")
print(f"Bottom-Up Normalized   : DM = {dm_norm:7.4f} | p-value = {p_norm:.4f}")
print(f"Bottom-Up Unnormalized : DM = {dm_unnorm:7.4f} | p-value = {p_unnorm:.4f}")

# 7. Export unified comparison file
df_eval.to_csv("comparison_results_final.csv")
print("\nSaved aligned 3-model comparison to 'comparison_results_final.csv'")

Model Approach                   |      RMSE |     MAPE |      MDA
1. Top-Down (Baseline)           |    227.50 |    0.70% |   51.95%
2. Bottom-Up (Normalized)        |    206.53 |    0.64% |   47.03%
3. Bottom-Up (Unnormalized)      |    250.20 |    0.80% |   48.05%
Aligned Out-of-Sample Trading Days: 691

--- DIEBOLD-MARIANO SIGNIFICANCE TESTS (vs. Top-Down Baseline) ---
Bottom-Up Normalized   : DM = -4.6371 | p-value = 0.0000
Bottom-Up Unnormalized : DM =  3.4605 | p-value = 0.0006

Saved aligned 3-model comparison to 'comparison_results_final.csv'
